# 02: Recommender Training

**Cold-Start Social-Graph Leakage: Simulation Study**

Reads the datasets that 01 saved to Drive; trains **standard, not-designed-to-leak** recommender models on them; **saves** the trained embeddings and weights **to Drive.** The next notebook (03: Attacker) will read these and perform the cold→warm scan.

### Design breaking the circularity (circularity): why is this not rigged?
We build two canonical architectures:
- **LightGCN**: graph-only structural embedding (parameter-free neighborhood propagation + BPR).
- **FeatureGNN**: a GraphSAGE/GraFRank-lite that combines structure **+** node features (multi-modal fusion).

Both are trained for the standard **link-prediction** objective. We do not hardcode a *"recommend the target's neighbor"* rule anywhere. The leakage will instead **emerge on its own**, later, from the following standard behavior:

> When the observer account is not recognized (cold-start), the system reasonably falls back to **context**, that is, "accounts similar to the profile you viewed" (context-aware / session-based recommendation). Because the context profile is the target, a competent relevance model will inevitably rank the target's graph neighbors higher.

We use `link-AUC` to show that the model is **genuinely a competent recommender system** (not rigged/broken); the fact that the warm-start control comes out ≈ random proves that the leakage is specific to the cold/context condition.

## 0. Setup and connecting Drive

In [ ]:
!pip install -q networkx numpy pandas scipy scikit-learn matplotlib
# torch comes pre-installed in Colab; no need to install it separately.

In [ ]:
# --- repository-root anchor (added when notebooks moved into notebooks/) ---
# Notebooks live in notebooks/ but read and write cold_start_leakage/ at the
# repository root, so resolve that root explicitly instead of relying on cwd.
import os as _os_anchor
_here = _os_anchor.getcwd()
if _os_anchor.path.basename(_here) == 'notebooks':
    _os_anchor.chdir(_os_anchor.path.dirname(_here))
# --- end anchor ---
import os, json, pickle
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
import matplotlib.pyplot as plt

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')   # Non-Colab (local test)

DATA_DIR   = os.path.join(ROOT, 'data')
MODELS_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # Important in larger real-world data like Twitter/email
print("Data source :", DATA_DIR)
print("Model output:", MODELS_DIR)
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Read the dataset from Drive (the schema from 01, self-sufficient)

In [ ]:
class SocialGraphDataset:
    def __init__(s, name, G, private, community, node_list, features, meta=None):
        s.name, s.G, s.private, s.community = name, G, private, community
        s.node_list, s.features, s.meta = node_list, features, (meta or {})
    def followers(s, t):  return set(s.G.predecessors(t))
    def following(s, t):  return set(s.G.successors(t))
    def neighborhood(s, t): return s.followers(t) | s.following(t)   # N(t) = GROUND TRUTH

def load_dataset(name, root=DATA_DIR):
    d = os.path.join(root, name)
    G = pickle.load(open(os.path.join(d, "graph.pkl"), "rb"))
    ndf = pd.read_csv(os.path.join(d, "nodes.csv"))
    X = np.load(os.path.join(d, "features.npy"))
    return SocialGraphDataset(name, G, dict(zip(ndf.node_id, ndf["private"])),
                              dict(zip(ndf.node_id, ndf.community)), list(ndf.node_id), X)

## 2. Normalize adjacency + standard architectures

In [ ]:
def norm_adj(G, n):
    und = G.to_undirected()
    edges = list(und.edges())
    A = torch.zeros((n, n))
    if edges:
        eu, ev = zip(*edges)
        eu = torch.tensor(eu, dtype=torch.long); ev = torch.tensor(ev, dtype=torch.long)
        A[eu, ev] = 1.0; A[ev, eu] = 1.0            # vektorize (eski: Python donguisiyle tek tek)
    A += torch.eye(n)                                  # self-loop
    deg = A.sum(1); dinv = torch.pow(deg, -0.5); dinv[torch.isinf(dinv)] = 0
    Ahat = A * dinv.unsqueeze(1) * dinv.unsqueeze(0)     # D^-1/2 (A+I) D^-1/2 by broadcasting (not a dense diagonal matmul)
    return Ahat.to(DEVICE)

class LightGCN(nn.Module):
    # single-graph: learnable base embedding + parameter-free propagation
    def __init__(self, n, dim=64, layers=3):
        super().__init__()
        self.emb = nn.Embedding(n, dim); nn.init.normal_(self.emb.weight, std=0.1)
        self.layers = layers
    def forward(self, Ahat):
        e = self.emb.weight; outs = [e]
        for _ in range(self.layers):
            e = Ahat @ e; outs.append(e)
        return torch.stack(outs, 0).mean(0)             # layer combination (mean)

class FeatureGNN(nn.Module):
    # GraphSAGE/GraFRank-lite: identity embedding + node features (multi-modal fusion)
    def __init__(self, n, feat_dim, dim=64, layers=2):
        super().__init__()
        self.id_emb = nn.Embedding(n, dim); nn.init.normal_(self.id_emb.weight, std=0.1)
        self.feat_proj = nn.Linear(feat_dim, dim)
        self.W = nn.ModuleList([nn.Linear(2*dim, dim) for _ in range(layers)])
        self.act = nn.ReLU(); self.layers = layers
    def forward(self, Ahat, Xfeat):
        h = self.id_emb.weight + self.feat_proj(Xfeat)
        for l in range(self.layers):
            agg = Ahat @ h
            h = self.act(self.W[l](torch.cat([h, agg], 1)))
        return h

## 3. BPR training + competence test (link-AUC)

**BPR** (Bayesian Personalized Ranking) standard ranking objective: a real link (positive) should receive a higher score than a random pair (negative). **link-AUC**, on the other hand, shows that the model actually works, that it can distinguish neighbors from non-neighbors: proof of our claim that this is "not a cheat, but a competent recommendation system."

In [ ]:
def train_bpr(model, Ahat, pos_pairs, n_nodes, Xfeat=None, epochs=60, bs=1024, lr=0.01):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos = torch.tensor(pos_pairs, dtype=torch.long, device=DEVICE)
    for ep in range(epochs):
        perm = torch.randperm(pos.size(0), device=DEVICE); tot = 0.0
        for i in range(0, pos.size(0), bs):
            idx = perm[i:i+bs]; u, p = pos[idx, 0], pos[idx, 1]
            neg = torch.randint(0, n_nodes, (u.size(0),), device=DEVICE)
            E = model(Ahat) if Xfeat is None else model(Ahat, Xfeat)
            su = (E[u]*E[p]).sum(1); sn = (E[u]*E[neg]).sum(1)
            loss = -torch.log(torch.sigmoid(su - sn) + 1e-9).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if ep % 20 == 0 or ep == epochs-1:
            print(f"  epoch {ep:3d}  bpr_loss={tot:.3f}")
    with torch.no_grad():
        return (model(Ahat) if Xfeat is None else model(Ahat, Xfeat)).detach().cpu()

def link_auc(E, G, n_samples=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    und = G.to_undirected(); edges = list(und.edges()); nodes = list(G.nodes())
    pos = [edges[i] for i in rng.choice(len(edges), size=min(n_samples, len(edges)), replace=False)]
    En = torch.nn.functional.normalize(E, dim=1)
    sc = lambda a, b: float((En[a]*En[b]).sum())
    ps = np.array([sc(u, v) for u, v in pos]); ns = []
    while len(ns) < len(ps):
        a, b = rng.choice(nodes, 2, replace=False)
        if not und.has_edge(a, b): ns.append(sc(a, b))
    ns = np.array(ns)
    return float((ps[:, None] > ns[None, :]).mean())    # P(pos > neg)

## 4. Train the Models and Save to Drive

In [ ]:
DATASETS = ["sbm_default", "lfr_default", "twitter_real", "email_real"]      # outputs of 01 (+ Reviewer 3.1)
EPOCHS = 60
rows = []

for name in DATASETS:
    try:
        ds = load_dataset(name)
    except Exception as e:
        print(f"[skipped] {name}: {e}"); continue
    n = len(ds.node_list)
    Ahat = norm_adj(ds.G, n)
    Xfeat = torch.tensor(ds.features, dtype=torch.float32, device=DEVICE)
    pos_pairs = [(u, v) for u, v in ds.G.to_undirected().edges()]

    print(f"\n=== {name} | LightGCN (graph-only) ===")
    lg = LightGCN(n, dim=64, layers=3)
    E_lg = train_bpr(lg, Ahat, pos_pairs, n, epochs=EPOCHS)
    auc_lg = link_auc(E_lg, ds.G)

    print(f"=== {name} | FeatureGNN (graph+feature) ===")
    fg = FeatureGNN(n, ds.features.shape[1], dim=64, layers=2)
    E_fg = train_bpr(fg, Ahat, pos_pairs, n, Xfeat=Xfeat, epochs=EPOCHS)
    auc_fg = link_auc(E_fg, ds.G)

    for mdl_name, E, mdl, auc in [("lightgcn", E_lg, lg, auc_lg),
                                  ("featuregnn", E_fg, fg, auc_fg)]:
        np.save(os.path.join(MODELS_DIR, f"{name}__{mdl_name}__emb.npy"), E.numpy())
        torch.save(mdl.state_dict(), os.path.join(MODELS_DIR, f"{name}__{mdl_name}__state.pt"))
        json.dump({"dataset": name, "model": mdl_name, "dim": 64, "epochs": EPOCHS,
                   "link_auc": round(auc, 4)},
                  open(os.path.join(MODELS_DIR, f"{name}__{mdl_name}__meta.json"), "w"), indent=2)
        rows.append({"dataset": name, "model": mdl_name, "link_auc": round(auc, 3)})

pd.DataFrame(rows)

## 5. Cold→warm gated scoring: *mechanism*

The "suggested accounts" generated when an observer account visits a target profile:

$$\text{score}(c \mid \text{observer}, \text{target}) = \alpha(h)\cdot \text{sim}(e_{\text{observer}}, e_c) + (1-\alpha(h))\cdot \text{sim}(e_{\text{target}}, e_c)$$

- $e_{\text{observer}}$ : the average of the embeddings of the observer's **h** known connections (standard inductive aggregation).
- $\alpha(h) = h/(h+k)$ : the **confidence** placed in the observer's own signal. $h=0 \Rightarrow \alpha=0 \Rightarrow$ pure **context** (target) fallback.

> There is **no** rule here such as *"suggest the target's neighbors."* Only (i) a standard confidence gate and (ii) relevance to the context profile exist. The leakage arises as an **emergent** effect from the intersection of these.

In [ ]:
class Recommender:
    def __init__(self, E, node_list, k_gate=5):
        self.E = torch.nn.functional.normalize(torch.as_tensor(E, dtype=torch.float32), dim=1)
        self.node_list = node_list; self.k_gate = k_gate
    def alpha(self, h): return h / (h + self.k_gate)
    def observer_embedding(self, contacts):
        if len(contacts) == 0: return None
        return torch.nn.functional.normalize(self.E[list(contacts)].mean(0, keepdim=True), dim=1)[0]
    def suggest(self, target, observer_contacts, k=20, exclude=None):
        h = len(observer_contacts); a = self.alpha(h)
        ctx = self.E @ self.E[target]                          # context (target) relevance
        if h > 0:
            obs = self.E @ self.observer_embedding(observer_contacts)
            score = a*obs + (1-a)*ctx
        else:
            score = ctx                                        # cold: saf fallback
        score = score.clone(); score[target] = -1e9
        for x in (exclude or []): score[x] = -1e9
        return torch.topk(score, k).indices.tolist()

## 6. Mechanism demo: does the leakage emerge *on its own*?

We scan the same targets with the same model using two observer types: **cold** (h=0, no history) and **warm** (unrelated to the target, with its own connections). Cold precision is expected to come out high, warm's ≈ random. *(Full scan and calibration in 03.)*

In [ ]:
demo = "sbm_default"
ds = load_dataset(demo)
E = np.load(os.path.join(MODELS_DIR, f"{demo}__lightgcn__emb.npy"))
rec = Recommender(E, ds.node_list, k_gate=5)
rng = np.random.default_rng(1)
allnodes = list(ds.G.nodes())
targets = [t for t in ds.node_list if ds.private[t] and ds.G.degree(t) >= 4]

def precision(t, contacts, k=20):
    sugg = rec.suggest(t, contacts, k=k)
    return len(set(sugg) & ds.neighborhood(t)) / k

cold, warm = [], []
for t in targets:
    cold.append(precision(t, set(), k=20))
    wc = set(rng.choice(allnodes, 15, replace=False)) - ds.neighborhood(t) - {t}
    warm.append(precision(t, wc, k=20))

print(f"{demo} | {len(targets)} private targets | LightGCN")
print(f"  COLD precision@20 = {np.mean(cold):.3f}")
print(f"  WARM precision@20 = {np.mean(warm):.3f}")

plt.figure(figsize=(5,4))
plt.bar(["cold (h=0)", "warm"], [np.mean(cold), np.mean(warm)],
        color=["#C0504D", "#4F81BD"], yerr=[np.std(cold), np.std(warm)], capsize=6)
plt.ylabel("precision@20 (intersection with N(t))")
plt.title("Leakage emergent: cold ≫ warm"); plt.tight_layout(); plt.show()

## For the next notebook

**04 Attacker Metrics** will read the following artifacts from Drive:
- `models/<veri>__<model>__emb.npy` (embeddings): `Recommender` is built with these,
- `data/<veri>/...` (for ground-truth `N(t)`).

To be done in 03: **dose-response curve** by sweeping the observer's history budget **h** from 0→large; **precision/recall/F1** over all private targets; **attacker advantage** against random/popularity/same-community baselines; and **calibration** to the observed ~72.6% cold-start precision.